# Continuous ability x relative-time representation with uncertainty (C2, C3)

**Objective.** Build the uncertainty-aware continuous representation (ability from an IRT model with guessing; relative time adjusted for ability, booklet, items reached and item time intensities) and compare it with discrete score-time profiles: masked-item prediction, calibration, predictive intervals, split-half stability, survey weights, self-reported effort. Development used the Form A training split; confirmation used the untouched Form B and C test splits (protocol: reports/protocol_continuous_temporal.md).

**Data.** Real mode needs the PISA 2025 public-use files processed as described in `data/README.md` (licensed by the OECD and never committed). For Colab, upload `data/processed/math_form*.parquet/.json` and `student_external.parquet` to the same relative paths, or mount Google Drive. Demo mode generates synthetic data. **Demo outputs are not findings.** The *saved results* cells display the PISA 2025 aggregate outputs committed in `results/`.

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
REPO_URL = ""   # GitHub repository; a private repository needs a token in the URL or an uploaded copy
DEMO = True            # True = synthetic demonstration (no PISA data needed); False = real PISA analysis
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        if not REPO_URL:
            raise SystemExit("Set REPO_URL or upload the repository folder to /content.")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
# 1. Synthetic demonstration (runs anywhere, no PISA data): fit the representation and look at it
from mcq_profiles.dashboard import simulate_demo, compute_map, LIMITATIONS
demo = simulate_demo(n=1500, J=20, seed=0)
co, summ = compute_map(demo, level=0.9)
print(summ); co.head()

In [ ]:
import matplotlib.pyplot as plt
s = co.dropna(subset=['rel_time']).sample(300, random_state=0)
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(s.theta, s.rel_time, xerr=[s.theta - s.theta_lo, s.theta_hi - s.theta], yerr=[s.rel_time - s.rel_time_lo, s.rel_time_hi - s.rel_time],
            fmt='o', ms=3, color='#2a78d6', ecolor='#c8d9f2', elinewidth=0.8)
ax.axhline(0, color='#52514e', ls='--', lw=1); ax.set(xlabel='Ability (logits)', ylabel='Relative time (SD)', title='SYNTHETIC demo')
plt.show()
print('
'.join(LIMITATIONS))

In [ ]:
# 2. Real analysis (needs the PISA extracts; about 10-40 min per form)
import subprocess, sys
if not DEMO:
    for args in [['--part', 'dev'], ['--part', 'confirm', '--form', 'math_formB'], ['--part', 'confirm', '--form', 'math_formC']]:
        r = subprocess.run([sys.executable, 'scripts/run_continuous_eval.py'] + args, capture_output=True, text=True)
        print(r.stdout[-3000:]); r.check_returncode()

In [ ]:
# 3. Saved PISA 2025 results (aggregate outputs committed to the repository)
import json, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, Image
R = Path('results/continuous')
for part, form in [('dev', 'math_formA'), ('confirm', 'math_formB'), ('confirm', 'math_formC')]:
    f = R / f'{part}_{form}_scores.csv'
    if not f.exists():
        print('missing', f); continue
    display(Markdown(f'**{part}, {form}**'))
    display(pd.read_csv(f)); display(pd.read_csv(R / f'{part}_{form}_paired.csv'))
    print(json.dumps(json.loads((R / f'{part}_{form}_summary.json').read_text()), indent=1)[:3000])

In [ ]:
for f in ['fig16_continuous_eval.png']:
    p = Path('results/figures') / f
    display(Image(str(p))) if p.exists() else print('missing', p)

In [ ]:
# 4. Streamlit prototype: run locally with  streamlit run app/streamlit_app.py
# (in Colab, Streamlit needs a tunnel; the back-end used above is the same code: mcq_profiles.dashboard)